# Employee Attrition Prediction - Modeling

This notebook trains and compares machine learning models for employee attrition prediction. Because attrition is an imbalanced classification problem, the model comparison focuses on recall for the attrition class, ROC-AUC, and interpretability rather than accuracy alone.

## 1. Import Libraries

In [0]:
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

## 2. Load Reusable Databricks Table

In [0]:
spark_df = spark.table("attrition_project.employee_attrition")

df = spark_df.toPandas()

print(f"Rows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")

display(df.head())

Rows: 1470
Columns: 35


Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,EnvironmentSatisfaction,Gender,HourlyRate,JobInvolvement,JobLevel,JobRole,JobSatisfaction,MaritalStatus,MonthlyIncome,MonthlyRate,NumCompaniesWorked,Over18,OverTime,PercentSalaryHike,PerformanceRating,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,2,Female,94,3,2,Sales Executive,4,Single,5993,19479,8,Y,Yes,11,3,1,80,0,8,0,1,6,4,0,5
49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,3,Male,61,2,2,Research Scientist,2,Married,5130,24907,1,Y,No,23,4,4,80,1,10,3,3,10,7,1,7
37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,4,Male,92,2,1,Laboratory Technician,3,Single,2090,2396,6,Y,Yes,15,3,2,80,0,7,3,3,0,0,0,0
33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,5,4,Female,56,3,1,Research Scientist,3,Married,2909,23159,1,Y,Yes,11,3,3,80,0,8,3,3,8,7,3,0
27,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,7,1,Male,40,3,1,Laboratory Technician,2,Married,3468,16632,9,Y,No,12,3,4,80,1,6,3,3,2,2,2,2


## 3. Prepare Features and Target

In [0]:
DROP_COLUMNS = ["EmployeeCount", "EmployeeNumber", "Over18", "StandardHours"]
TARGET = "Attrition"
RANDOM_STATE = 42

# Drop non-informative columns
df_model = df.drop(columns=[col for col in DROP_COLUMNS if col in df.columns])

# Convert target variable from Yes/No to 1/0
y = df_model[TARGET].map({"No": 0, "Yes": 1})

# Keep all remaining columns as features
X = df_model.drop(columns=[TARGET])

print(f"Feature columns: {X.shape[1]}")
print("Target distribution:")
print(y.value_counts(normalize=True).rename("percentage").mul(100).round(2))

Feature columns: 30
Target distribution:
Attrition
0    83.88
1    16.12
Name: percentage, dtype: float64


This confirms:

- We removed 4 non-informative columns.
- We converted Attrition into binary target.
- We have 30 feature columns.
- The target is imbalanced, because only 16.12% are attrition cases.

## 4. Train/Test Split

In [0]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    stratify=y,
    random_state=RANDOM_STATE,
)

print(f"Training rows: {X_train.shape[0]}")
print(f"Testing rows: {X_test.shape[0]}")

print("\nTraining target distribution:")
print(y_train.value_counts(normalize=True).rename("percentage").mul(100).round(2))

print("\nTesting target distribution:")
print(y_test.value_counts(normalize=True).rename("percentage").mul(100).round(2))

Training rows: 1029
Testing rows: 441

Training target distribution:
Attrition
0    83.87
1    16.13
Name: percentage, dtype: float64

Testing target distribution:
Attrition
0    83.9
1    16.1
Name: percentage, dtype: float64


This proves stratify=y worked correctly. The train and test sets keep the same attrition imbalance.

## 5. Build Preprocessing Pipeline

In [0]:
numeric_features = X_train.select_dtypes(include="number").columns.tolist()
categorical_features = X_train.select_dtypes(exclude="number").columns.tolist()

print(f"Numeric features: {len(numeric_features)}")
print(f"Categorical features: {len(categorical_features)}")

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", StandardScaler(), numeric_features),
        ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ]
)

Numeric features: 23
Categorical features: 7


## 6. Train Candidate Models

In [0]:
models = {
    "logistic_regression": LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=RANDOM_STATE,
    ),
    "random_forest": RandomForestClassifier(
        n_estimators=300,
        class_weight="balanced",
        min_samples_leaf=5,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
    "gradient_boosting": GradientBoostingClassifier(
        random_state=RANDOM_STATE,
    ),
}

results = []
fitted_pipelines = {}

for model_name, estimator in models.items():
    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", estimator),
        ]
    )

    pipeline.fit(X_train, y_train)

    predicted = pipeline.predict(X_test)
    probabilities = pipeline.predict_proba(X_test)[:, 1]

    report = classification_report(
        y_test,
        predicted,
        output_dict=True,
        zero_division=0,
    )

    results.append(
        {
            "model": model_name,
            "accuracy": report["accuracy"],
            "attrition_precision": report["1"]["precision"],
            "attrition_recall": report["1"]["recall"],
            "attrition_f1": report["1"]["f1-score"],
            "roc_auc": roc_auc_score(y_test, probabilities),
        }
    )

    fitted_pipelines[model_name] = pipeline

    print(f"\n{model_name}")
    print(confusion_matrix(y_test, predicted))
    print(classification_report(y_test, predicted, zero_division=0))


logistic_regression
[[289  81]
 [ 23  48]]
              precision    recall  f1-score   support

           0       0.93      0.78      0.85       370
           1       0.37      0.68      0.48        71

    accuracy                           0.76       441
   macro avg       0.65      0.73      0.66       441
weighted avg       0.84      0.76      0.79       441


random_forest
[[343  27]
 [ 48  23]]
              precision    recall  f1-score   support

           0       0.88      0.93      0.90       370
           1       0.46      0.32      0.38        71

    accuracy                           0.83       441
   macro avg       0.67      0.63      0.64       441
weighted avg       0.81      0.83      0.82       441


gradient_boosting
[[355  15]
 [ 54  17]]
              precision    recall  f1-score   support

           0       0.87      0.96      0.91       370
           1       0.53      0.24      0.33        71

    accuracy                           0.84       441
   m

In [0]:
results_df = pd.DataFrame(results).sort_values(
    ["attrition_recall", "roc_auc"],
    ascending=False,
)

display(results_df)

model,accuracy,attrition_precision,attrition_recall,attrition_f1,roc_auc
logistic_regression,0.764172335600907,0.37209302325581395,0.676056338028169,0.48,0.8162923486867149
random_forest,0.8299319727891157,0.46,0.323943661971831,0.38016528925619836,0.7738104301484583
gradient_boosting,0.8435374149659864,0.53125,0.23943661971830985,0.3300970873786408,0.7835173201370385


Logistic Regression was selected as the preferred model because it achieved the highest recall for the attrition class. In this business case, recall is important because the goal is to identify employees who may be at risk of leaving. Although Gradient Boosting achieved higher overall accuracy, it missed more attrition cases, making it less useful for retention monitoring.

## 7. Save Model Metrics Table

This step saves the model comparison results as a reusable Databricks table so the dashboard notebook can use the modeling output without retraining the models.

In [0]:
spark_results_df = spark.createDataFrame(results_df)

spark_results_df.write.mode("overwrite").saveAsTable(
    "attrition_project.model_metrics"
)

In [0]:
display(spark.table("attrition_project.model_metrics"))

model,accuracy,attrition_precision,attrition_recall,attrition_f1,roc_auc
logistic_regression,0.764172335600907,0.37209302325581395,0.676056338028169,0.48,0.8162923486867149
random_forest,0.8299319727891157,0.46,0.323943661971831,0.38016528925619836,0.7738104301484583
gradient_boosting,0.8435374149659864,0.53125,0.23943661971830985,0.3300970873786408,0.7835173201370385


## 8. Final Modeling Summary

The modeling results show that Logistic Regression is the best choice for this project because it provides the highest recall for employees who left the company. This makes it useful for retention monitoring, where missing at-risk employees can be more costly than false alarms. The model is also easier to explain to HR and business stakeholders compared with more complex models.